# Kaggle 10. Russian Baselines: ruRoBERTa and SBERT/RoSBERTa

Цель ноутбука: закрыть замечание рецензентов про ограниченный набор
русскоязычных baseline-моделей.

Запускаются две разные парадигмы:

1. `ruRoBERTa` fine-tuning: supervised encoder baseline.
2. `SBERT/RoSBERTa + LogisticRegression`: компактный embedding baseline.

По умолчанию используется обычный русский текст `text_ru`, не
маскированный текст и не explanations. Это важно: сравнение честно
проверяет именно text-only baseline на русскоязычном корпусе.

In [ ]:
%pip install -q -U transformers sentencepiece accelerate scikit-learn sentence-transformers seaborn

In [ ]:
import copy
import json
import logging
import random
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("kaggle_ru_baselines")
sns.set_theme(style="whitegrid")

In [ ]:
INPUT_JSONL_CANDIDATES = sorted(Path("/kaggle/input").rglob("*.jsonl"))
assert INPUT_JSONL_CANDIDATES, "Upload COCOLOFA-RU JSONL dataset to /kaggle/input first."

for path in INPUT_JSONL_CANDIDATES:
    print(path)

DATASET_PATH = (
    next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2_public.jsonl"), None)
    or next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2_explanations.jsonl"), None)
    or next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2.jsonl"), None)
    or next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2_masked.jsonl"), None)
)
assert DATASET_PATH is not None, "No supported COCOLOFA-RU dataset was found."

OUTPUT_ROOT = Path("/kaggle/working/phase3_ru_baselines")
TEXT_COLUMN = "text_ru"
SEED = 42

RUN_RUROBERTA = True
RUN_SBERT_BASELINES = True

RUROBERTA_MODEL_NAME = "ai-forever/ruRoberta-large"
SBERT_MODEL_NAMES = [
    "ai-forever/ru-en-RoSBERTa",
    # Faster fallback if Kaggle has issues with the model above:
    # "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

# First debug run can use small values, final paper run should use None.
TRAIN_PER_CLASS_LIMIT = None
DEV_PER_CLASS_LIMIT = None
TEST_PER_CLASS_LIMIT = None

MAX_LENGTH = 256
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EARLY_STOPPING_PATIENCE = 2
SAVE_BEST_RUROBERTA = True

{
    "dataset_path": str(DATASET_PATH),
    "output_root": str(OUTPUT_ROOT),
    "text_column": TEXT_COLUMN,
    "ruroberta_model": RUROBERTA_MODEL_NAME,
    "sbert_models": SBERT_MODEL_NAMES,
    "seed": SEED,
    "max_length": MAX_LENGTH,
}

In [ ]:
LABELS = [
    "none",
    "appeal to authority",
    "appeal to majority",
    "appeal to nature",
    "appeal to tradition",
    "appeal to worse problems",
    "false dilemma",
    "hasty generalization",
    "slippery slope",
]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
ACCEPTED_TRANSLATION_STATUSES = {"ok", "repaired_ok"}


def normalize_label(label: str) -> str:
    return " ".join(str(label).strip().lower().split())


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def parse_json_records(path: Path) -> list[dict]:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []
    rows = []
    decoder = json.JSONDecoder()
    cursor = 0
    while cursor < len(text):
        while cursor < len(text) and text[cursor].isspace():
            cursor += 1
        if cursor >= len(text):
            break
        parsed, next_cursor = decoder.raw_decode(text, cursor)
        if isinstance(parsed, list):
            rows.extend(parsed)
        else:
            rows.append(parsed)
        cursor = next_cursor
        while cursor < len(text) and text[cursor] in ",\n\r\t ":
            cursor += 1
    return rows


def load_cocolofa_jsonl(path: Path, text_column: str) -> pd.DataFrame:
    df = pd.DataFrame(parse_json_records(path))
    required = {"sample_id", "label_str", "split", text_column}
    missing = sorted(required - set(df.columns))
    assert not missing, f"Missing required columns: {missing}"

    df = df.copy()
    df["label_str"] = df["label_str"].map(normalize_label)
    df = df[df["label_str"].isin(LABEL_TO_ID)].copy()
    df["label_id"] = df["label_str"].map(LABEL_TO_ID).astype(int)
    df[text_column] = df[text_column].fillna("").astype(str).str.strip()
    df = df[df[text_column].str.len() > 0].copy()

    if "translation_status" in df.columns:
        df["translation_status"] = df["translation_status"].fillna("ok").astype(str)
        df = df[df["translation_status"].isin(ACCEPTED_TRANSLATION_STATUSES)].copy()

    return df.sort_values("sample_id").reset_index(drop=True)


def sample_per_class(df: pd.DataFrame, limit: int | None, seed: int) -> pd.DataFrame:
    if limit is None:
        return df.copy()
    parts = []
    for label in LABELS:
        group = df[df["label_str"] == label]
        n = min(limit, len(group))
        parts.append(group.sample(n=n, random_state=seed) if n else group)
    return pd.concat(parts, ignore_index=True).sort_values(["label_id", "sample_id"]).reset_index(drop=True)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False))
            handle.write("\n")


def compute_metrics(y_true: list[int], y_pred: list[int]) -> dict:
    p, r, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(len(LABELS))),
        zero_division=0,
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(np.mean(p)),
        "macro_recall": float(np.mean(r)),
        "macro_f1": float(np.mean(f1)),
        "per_class": {
            LABELS[i]: {
                "precision": float(p[i]),
                "recall": float(r[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i in range(len(LABELS))
        },
    }


def show_confusion(y_true: list[int], y_pred: list[int], title: str) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS))))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Gold")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


set_global_seed(SEED)

In [ ]:
df = load_cocolofa_jsonl(DATASET_PATH, TEXT_COLUMN)

train_df = sample_per_class(df[df["split"] == "train"], TRAIN_PER_CLASS_LIMIT, SEED)
dev_df = sample_per_class(df[df["split"].isin(["dev", "valid", "validation"])], DEV_PER_CLASS_LIMIT, SEED)
test_df = sample_per_class(df[df["split"] == "test"], TEST_PER_CLASS_LIMIT, SEED)

assert len(train_df) and len(dev_df) and len(test_df), {
    "train": len(train_df),
    "dev": len(dev_df),
    "test": len(test_df),
}

print("rows:", len(df))
print("split distribution:", dict(df["split"].value_counts()))
display(df["label_str"].value_counts().reindex(LABELS).rename("all_rows").to_frame())
display(train_df["label_str"].value_counts().reindex(LABELS).rename("train").to_frame())
display(dev_df["label_str"].value_counts().reindex(LABELS).rename("dev").to_frame())
display(test_df["label_str"].value_counts().reindex(LABELS).rename("test").to_frame())
display(df[["sample_id", "split", "label_str", TEXT_COLUMN]].head(3))

## SBERT / RoSBERTa embedding baseline

Этот baseline не дообучает трансформер. Он строит sentence embeddings
для `text_ru`, затем обучает простую `LogisticRegression`.

Это хороший компактный baseline для ответа рецензенту: он показывает,
насколько далеко можно зайти без полноценного fine-tuning-а encoder-а.

In [ ]:
def run_sbert_baseline(model_name: str) -> dict:
    output_dir = OUTPUT_ROOT / "sbert" / model_name.replace("/", "__")
    output_dir.mkdir(parents=True, exist_ok=True)

    encoder = SentenceTransformer(model_name)
    train_x = encoder.encode(
        train_df[TEXT_COLUMN].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    dev_x = encoder.encode(
        dev_df[TEXT_COLUMN].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    test_x = encoder.encode(
        test_df[TEXT_COLUMN].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    train_y = train_df["label_id"].tolist()
    dev_y = dev_df["label_id"].tolist()
    test_y = test_df["label_id"].tolist()

    rows = []
    best = None
    for c_value in [0.25, 0.5, 1.0, 2.0, 4.0]:
        clf = LogisticRegression(
            C=c_value,
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs",
            n_jobs=-1,
            random_state=SEED,
        )
        clf.fit(train_x, train_y)
        dev_pred = clf.predict(dev_x).tolist()
        dev_metrics = compute_metrics(dev_y, dev_pred)
        row = {"C": c_value, **{f"dev_{k}": v for k, v in dev_metrics.items() if k != "per_class"}}
        rows.append(row)
        if best is None or dev_metrics["macro_f1"] > best["dev_metrics"]["macro_f1"]:
            best = {"C": c_value, "clf": clf, "dev_metrics": dev_metrics}

    assert best is not None
    test_pred = best["clf"].predict(test_x).tolist()
    test_metrics = compute_metrics(test_y, test_pred)

    predictions = []
    for row, pred_id in zip(test_df.to_dict("records"), test_pred):
        predictions.append(
            {
                "sample_id": row["sample_id"],
                "gold_label": row["label_str"],
                "pred_label": ID_TO_LABEL[int(pred_id)],
                "correct": row["label_id"] == int(pred_id),
            }
        )

    summary = {
        "method": "sbert_logreg",
        "model_name": model_name,
        "text_column": TEXT_COLUMN,
        "best_C": best["C"],
        "dev_metrics": best["dev_metrics"],
        "test_metrics": test_metrics,
        "output_dir": str(output_dir),
    }
    pd.DataFrame(rows).to_csv(output_dir / "dev_grid.csv", index=False)
    write_json(output_dir / "summary.json", summary)
    write_jsonl(output_dir / "test_predictions.jsonl", predictions)

    print(model_name, "best C:", best["C"])
    display(pd.DataFrame(rows))
    print("TEST:", json.dumps(test_metrics, ensure_ascii=False, indent=2))
    show_confusion(test_y, test_pred, f"SBERT baseline: {model_name}")
    return summary


sbert_summaries = []
if RUN_SBERT_BASELINES:
    for model_name in SBERT_MODEL_NAMES:
        sbert_summaries.append(run_sbert_baseline(model_name))
sbert_summaries

## ruRoBERTa fine-tuning baseline

Это supervised baseline: модель-кодировщик дообучается на train split,
выбирается лучшая эпоха по dev macro-F1, test используется один раз
для финальной оценки.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer: AutoTokenizer, text_column: str, max_length: int):
        self.rows = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_column = text_column
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        row = self.rows.iloc[idx]
        encoded = self.tokenizer(
            str(row[self.text_column]),
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item["labels"] = torch.tensor(int(row["label_id"]), dtype=torch.long)
        return item


@torch.no_grad()
def evaluate_model(model, loader: DataLoader, device: torch.device) -> tuple[dict, list[int], list[int]]:
    model.eval()
    gold, pred = [], []
    for batch in tqdm(loader, desc="Eval", leave=False):
        labels = batch.pop("labels").to(device)
        batch = {key: value.to(device) for key, value in batch.items()}
        logits = model(**batch).logits
        pred.extend(torch.argmax(logits, dim=-1).detach().cpu().tolist())
        gold.extend(labels.detach().cpu().tolist())
    return compute_metrics(gold, pred), gold, pred


def class_weights_from_frame(frame: pd.DataFrame) -> torch.Tensor:
    counts = frame["label_id"].value_counts().reindex(range(len(LABELS)), fill_value=1).sort_index()
    weights = 1.0 / np.sqrt(counts.to_numpy(dtype=np.float32))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def run_ruroberta_baseline() -> dict:
    output_dir = OUTPUT_ROOT / "ruroberta" / RUROBERTA_MODEL_NAME.replace("/", "__")
    best_model_dir = output_dir / "best_model"
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    tokenizer = AutoTokenizer.from_pretrained(RUROBERTA_MODEL_NAME, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        RUROBERTA_MODEL_NAME,
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
    ).to(device)

    train_loader = DataLoader(
        TextDataset(train_df, tokenizer, TEXT_COLUMN, MAX_LENGTH),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=device.type == "cuda",
    )
    dev_loader = DataLoader(
        TextDataset(dev_df, tokenizer, TEXT_COLUMN, MAX_LENGTH),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=device.type == "cuda",
    )
    test_loader = DataLoader(
        TextDataset(test_df, tokenizer, TEXT_COLUMN, MAX_LENGTH),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=device.type == "cuda",
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_update_steps = max(1, (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS)
    warmup_steps = int(total_update_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_update_steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_from_frame(train_df).to(device))
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_dev_f1 = -1.0
    best_epoch = None
    patience_left = EARLY_STOPPING_PATIENCE
    history = []

    epoch_bar = tqdm(range(1, NUM_EPOCHS + 1), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        seen_batches = 0

        batch_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False)
        for step, batch in enumerate(batch_bar, start=1):
            labels = batch.pop("labels").to(device)
            batch = {key: value.to(device) for key, value in batch.items()}

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(**batch).logits
                loss = loss_fn(logits, labels) / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS
            seen_batches += 1

            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            batch_bar.set_postfix(loss=f"{running_loss / max(seen_batches, 1):.4f}")

        dev_metrics, _, _ = evaluate_model(model, dev_loader, device)
        train_loss = running_loss / max(seen_batches, 1)
        history_row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **{f"dev_{k}": v for k, v in dev_metrics.items() if k != "per_class"},
        }
        history.append(history_row)

        if dev_metrics["macro_f1"] > best_dev_f1:
            best_dev_f1 = dev_metrics["macro_f1"]
            best_epoch = epoch
            patience_left = EARLY_STOPPING_PATIENCE
            if SAVE_BEST_RUROBERTA:
                model.save_pretrained(best_model_dir)
                tokenizer.save_pretrained(best_model_dir)
        else:
            patience_left -= 1

        epoch_bar.set_postfix(best_epoch=best_epoch, best_macro_f1=f"{best_dev_f1:.4f}")
        print(
            f"Epoch {epoch}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | "
            f"dev_macro_f1={dev_metrics['macro_f1']:.4f}"
        )
        if patience_left <= 0:
            print("Early stopping.")
            break

    if SAVE_BEST_RUROBERTA and best_model_dir.exists():
        model = AutoModelForSequenceClassification.from_pretrained(best_model_dir).to(device)

    test_metrics, test_gold, test_pred = evaluate_model(model, test_loader, device)
    predictions = []
    for row, pred_id in zip(test_df.to_dict("records"), test_pred):
        predictions.append(
            {
                "sample_id": row["sample_id"],
                "gold_label": row["label_str"],
                "pred_label": ID_TO_LABEL[int(pred_id)],
                "correct": row["label_id"] == int(pred_id),
            }
        )

    summary = {
        "method": "ruroberta_finetune",
        "model_name": RUROBERTA_MODEL_NAME,
        "text_column": TEXT_COLUMN,
        "best_epoch": best_epoch,
        "best_dev_macro_f1": best_dev_f1,
        "test_metrics": test_metrics,
        "output_dir": str(output_dir),
    }
    pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)
    write_json(output_dir / "summary.json", summary)
    write_jsonl(output_dir / "test_predictions.jsonl", predictions)

    print("TEST:", json.dumps(test_metrics, ensure_ascii=False, indent=2))
    show_confusion(test_gold, test_pred, f"ruRoBERTa: {RUROBERTA_MODEL_NAME}")
    return summary


ruroberta_summary = None
if RUN_RUROBERTA:
    ruroberta_summary = run_ruroberta_baseline()
ruroberta_summary

## Summary table

In [ ]:
summary_rows = []
for summary in sbert_summaries:
    row = {
        "method": summary["method"],
        "model": summary["model_name"],
        **summary["test_metrics"],
    }
    row.pop("per_class", None)
    summary_rows.append(row)

if ruroberta_summary is not None:
    row = {
        "method": ruroberta_summary["method"],
        "model": ruroberta_summary["model_name"],
        **ruroberta_summary["test_metrics"],
    }
    row.pop("per_class", None)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
display(summary_df)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(OUTPUT_ROOT / "summary_metrics.csv", index=False)